# Cross-Model ATR: GPT-2 Medium — Attractor Dominance & Basin Mapping

## Experiment Design
**This notebook is an EXACT replication of the GPT-2 Small ATR experiment (EXP_009d1),**
**applied to GPT-2 Medium to test whether attractor basin count scales with model capacity.**

### Key Difference: Model Scale (SAME training data)
- **GPT-2 Small:** 124M params, 12 layers, d_model=768
- **GPT-2 Medium:** 345M params, 24 layers, d_model=1024
- Both trained on **WebText** (Reddit outbound links, 2018)

### Hypotheses Under Test

**H_SC1: Basin Existence** — GPT-2 Medium will also exhibit discrete attractor basins under ATR.

**H_SC2: Basin Expansion** — With 2.8x more parameters and a larger d_model,
GPT-2 Medium may exhibit MORE basins than GPT-2 Small's five,
reflecting greater capacity to encode the same training corpus.

**H_SC3: Basin Overlap** — Since the training data is identical (WebText),
the basin *themes* should overlap with GPT-2 Small (political, theological, etc.),
even if specific tokens differ.

### Method
Identical to EXP_009d1. Same 125 prompts, same iteration schedule, same ATR engine.
Only the model loaded in STEP 1 is changed.

---


In [1]:
# ============================================================
# STEP 0: DEPENDENCIES
# ============================================================
import sys
!{sys.executable} -m pip install kaleido -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# STEP 1: CALIBRATION
# ============================================================
import torch
import numpy as np
import os
import plotly.graph_objects as go
import plotly.express as px
from transformer_lens import HookedTransformer
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-medium", device=device)
print(f"Architecture: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads, d_model={model.cfg.d_model}")
print(f"Running on: {device}")

# Output directory for all saved artifacts
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-medium into HookedTransformer
Architecture: 24 layers, 16 heads, d_model=1024
Running on: cpu
Output directory: c:\Users\Fab2\Desktop\AI\_learn\_fold\03_MECHINIPHYLUM\_LAB_NOTEBOOKS\lucier-repo\experiments\gpt2_medium\output


In [3]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', '..')))

# ============================================================
# STEP 2: CONFIGURATION — 125 Prompts from prompt_library.py
# ============================================================
from prompt_library import (
    PROMPT_LIBRARY, PREDICTIONS, CATEGORY_MAP,
    COMPLEX, NARRATIVE, SIMPLE, CHEMICAL, ACRONYMS, VULGARITY, WILD
)

# Tightened schedule: convergence occurs by ~100
ITERATION_SCHEDULE = [0, 2, 3, 5, 10, 20, 50, 100]
MAX_ITERATIONS = max(ITERATION_SCHEDULE)

LAYER_START = 0
LAYER_END = model.cfg.n_layers - 1

print(f"Schedule: {ITERATION_SCHEDULE}")
print(f"Room: Layers {LAYER_START} -> {LAYER_END}")
print(f"Total prompts: {len(PROMPT_LIBRARY)}")
print(f"\nBreakdown:")
for cat_name, cat_dict in [
    ("Complex", COMPLEX), ("Narrative", NARRATIVE),
    ("Simple", SIMPLE), ("Chemical", CHEMICAL),
    ("Acronyms", ACRONYMS), ("Vulgarity", VULGARITY),
    ("Wild", WILD)
]:
    print(f"  {cat_name}: {len(cat_dict)} prompts")

# Save config
config_md = f"""# Stage 1 Run Config\n
- **Prompts:** {len(PROMPT_LIBRARY)}\n
- **Schedule:** {ITERATION_SCHEDULE}\n
- **Layers:** {LAYER_START} -> {LAYER_END}\n
- **Device:** {device}\n
"""
with open(os.path.join(OUTPUT_DIR, 'config.md'), 'w', encoding='utf-8') as f:
    f.write(config_md)
print(f"\n[SAVED] {OUTPUT_DIR}/config.md")

Schedule: [0, 2, 3, 5, 10, 20, 50, 100]
Room: Layers 0 -> 23
Total prompts: 125

Breakdown:
  Complex: 25 prompts
  Narrative: 20 prompts
  Simple: 20 prompts
  Chemical: 10 prompts
  Acronyms: 10 prompts
  Vulgarity: 10 prompts
  Wild: 30 prompts

[SAVED] output/config.md


In [4]:
# ============================================================
# STEP 3: THE CORE ENGINE — Identical to lucier_total_resonance
# ============================================================

def get_top_tokens(model, resid_vector, k=5):
    """Decode a residual stream vector into top-k token predictions.
    Applies the Final LayerNorm before unembedding for correct decoding."""
    normalized = model.ln_final(resid_vector)
    logits = normalized @ model.W_U + model.b_U
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, k)
    tokens = [model.tokenizer.decode([idx]) for idx in top_indices]
    return list(zip(tokens, top_probs.tolist()))


def run_total_resonance_loop(model, prompt, layer_start, layer_end, max_iter, schedule):
    """
    TOTAL Lucier Loop: iteratively re-inject the ENTIRE residual stream
    tensor (all token positions) through the layer slice.
    Returns a list of snapshot dicts at each scheduled iteration.
    """
    snapshots = []
    hook_point_read = f"blocks.{layer_end}.hook_resid_post"
    hook_point_write = f"blocks.{layer_start}.hook_resid_pre"
    
    with torch.no_grad():
        _, cache = model.run_with_cache(
            prompt,
            names_filter=lambda n: n == hook_point_read
        )
    
    current_tensor = cache[hook_point_read][0].clone()
    seq_len = current_tensor.shape[0]
    initial_norm = current_tensor.norm().item()
    
    last_vec = current_tensor[-1, :].clone()
    mean_vec = current_tensor.mean(dim=0).clone()
    
    if 0 in schedule:
        top_tokens_last = get_top_tokens(model, last_vec)
        all_pos_tokens = []
        for pos in range(seq_len):
            pos_top = get_top_tokens(model, current_tensor[pos, :], k=1)
            all_pos_tokens.append(pos_top[0][0])
        snapshots.append({
            "iteration": 0,
            "tensor": current_tensor.clone().cpu(),
            "last_vector": last_vec.clone().cpu(),
            "mean_vector": mean_vec.clone().cpu(),
            "last_norm": last_vec.norm().item(),
            "mean_norm": mean_vec.norm().item(),
            "tensor_norm": current_tensor.norm().item(),
            "top_tokens": top_tokens_last,
            "all_position_tokens": all_pos_tokens,
            "cosine_sim_last": 1.0,
            "cosine_sim_mean": 1.0,
            "position_similarity": 1.0,
        })
    
    prev_last = last_vec.clone()
    prev_mean = mean_vec.clone()
    
    for i in range(1, max_iter + 1):
        # Normalise to maintain energy level
        current_norm = current_tensor.norm().item()
        if current_norm > 0:
            current_tensor = current_tensor * (initial_norm / current_norm)
        
        inject_tensor = current_tensor.clone()
        
        def injection_hook(resid, hook, tensor=inject_tensor):
            resid[0, :, :] = tensor
            return resid
        
        model.add_hook(hook_point_write, injection_hook)
        try:
            with torch.no_grad():
                _, cache = model.run_with_cache(
                    prompt,
                    names_filter=lambda n: n == hook_point_read
                )
        finally:
            model.reset_hooks()
        
        current_tensor = cache[hook_point_read][0].clone()
        last_vec = current_tensor[-1, :].clone()
        mean_vec = current_tensor.mean(dim=0).clone()
        
        if i in schedule:
            cos_sim_last = torch.nn.functional.cosine_similarity(
                last_vec.unsqueeze(0), prev_last.unsqueeze(0)
            ).item()
            cos_sim_mean = torch.nn.functional.cosine_similarity(
                mean_vec.unsqueeze(0), prev_mean.unsqueeze(0)
            ).item()
            
            pos_norms = current_tensor.norm(dim=1, keepdim=True).clamp(min=1e-8)
            normalized_positions = current_tensor / pos_norms
            pos_sim_matrix = normalized_positions @ normalized_positions.T
            mask = ~torch.eye(seq_len, dtype=torch.bool, device=pos_sim_matrix.device)
            position_similarity = pos_sim_matrix[mask].mean().item()
            
            top_tokens_last = get_top_tokens(model, last_vec)
            all_pos_tokens = []
            for pos in range(seq_len):
                pos_top = get_top_tokens(model, current_tensor[pos, :], k=1)
                all_pos_tokens.append(pos_top[0][0])
            
            snapshots.append({
                "iteration": i,
                "tensor": current_tensor.clone().cpu(),
                "last_vector": last_vec.clone().cpu(),
                "mean_vector": mean_vec.clone().cpu(),
                "last_norm": last_vec.norm().item(),
                "mean_norm": mean_vec.norm().item(),
                "tensor_norm": current_tensor.norm().item(),
                "top_tokens": top_tokens_last,
                "all_position_tokens": all_pos_tokens,
                "cosine_sim_last": cos_sim_last,
                "cosine_sim_mean": cos_sim_mean,
                "position_similarity": position_similarity,
            })
            print(f"  iter {i:>3}: top='{top_tokens_last[0][0].strip()}', "
                  f"cos_mean={cos_sim_mean:.4f}, pos_collapse={position_similarity:.4f}")
        
        prev_last = last_vec.clone()
        prev_mean = mean_vec.clone()
    
    return snapshots

print("Engine loaded.")

Engine loaded.


In [5]:
# ============================================================
# STEP 4: RUN ALL 125 PROMPTS
# ============================================================

all_results = {}

for idx, (label, prompt) in enumerate(PROMPT_LIBRARY.items()):
    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(PROMPT_LIBRARY)}] RECORDING: '{label}'")
    print(f"  Prompt: \"{prompt}\"")
    print(f"{'='*60}")
    
    snapshots = run_total_resonance_loop(
        model, prompt,
        layer_start=LAYER_START,
        layer_end=LAYER_END,
        max_iter=MAX_ITERATIONS,
        schedule=ITERATION_SCHEDULE
    )
    all_results[label] = snapshots
    
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    print(f"  y Terminal token: '{terminal}'")

print(f"\n{'='*60}")
print(f"ALL {len(all_results)} RECORDINGS COMPLETE.")


[1/125] RECORDING: 'A01_physics'
  Prompt: "The implications of quantum entanglement suggest that"
  iter   2: top='the', cos_mean=0.9908, pos_collapse=0.9398
  iter   3: top='local', cos_mean=0.9965, pos_collapse=0.9765
  iter   5: top='local', cos_mean=0.9993, pos_collapse=0.9947
  iter  10: top='D', cos_mean=1.0000, pos_collapse=0.9998
  iter  20: top='D', cos_mean=1.0000, pos_collapse=1.0000
  iter  50: top='D', cos_mean=1.0000, pos_collapse=1.0000
  iter 100: top='D', cos_mean=1.0000, pos_collapse=1.0000
  y Terminal token: 'D'

[2/125] RECORDING: 'A02_medical'
  Prompt: "A meta-analysis of randomised controlled trials indicates"
  iter   2: top='that', cos_mean=0.9913, pos_collapse=0.9645
  iter   3: top='that', cos_mean=0.9967, pos_collapse=0.9848
  iter   5: top='local', cos_mean=0.9994, pos_collapse=0.9954
  iter  10: top='D', cos_mean=1.0000, pos_collapse=0.9998
  iter  20: top='D', cos_mean=1.0000, pos_collapse=1.0000
  iter  50: top='D', cos_mean=1.0000, pos_collapse=1.000

---
## 5. Analysis

### 5a. Hypothesis Assessment — Predictions vs Actuals

In [6]:
# ============================================================
# VIS 5a: HYPOTHESIS ASSESSMENT — Predictions vs Actuals
# ============================================================

md = "# Stage 1 Results: Hypothesis Assessment\n\n"
md += "| Prompt | Category | Predicted | Actual Terminal | Match? |\n"
md += "|:---|:---|:---|:---|:---|\n"

basin_counts = {}
category_basins = {}
mismatches = []

for label, snapshots in all_results.items():
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    predicted_basin, confidence = PREDICTIONS[label]
    category = CATEGORY_MAP[label]
    
    # Classify actual basin
    # AFTER (exact match)
    if terminal == 'prolet':
        actual_basin = 'prolet'
    elif terminal == 'Divine':
        actual_basin = 'Divine'
    else:
        actual_basin = f'OTHER:{terminal}'
    
    basin_counts[actual_basin] = basin_counts.get(actual_basin, 0) + 1
    
    if category not in category_basins:
        category_basins[category] = []
    category_basins[category].append((label, actual_basin, terminal))
    
    match = 'y' if predicted_basin.lower() in actual_basin.lower() else 'n'
    if match == 'n' and predicted_basin != 'unknown':
        mismatches.append((label, predicted_basin, actual_basin))
    
    md += f"| {label} | {category} | `{predicted_basin}` ({confidence}) | `{terminal}` -> **{actual_basin}** | {match} |\n"

md += "\n---\n\n"
md += "## Basin Summary\n\n"
md += "| Basin | Count | % |\n"
md += "|:---|:---|:---|\n"
total = len(all_results)
for basin, count in sorted(basin_counts.items(), key=lambda x: -x[1]):
    md += f"| **{basin}** | {count} | {count/total*100:.1f}% |\n"

md += "\n---\n\n"
md += "## Category Breakdown\n\n"
for cat, entries in category_basins.items():
    md += f"### {cat} ({len(entries)} prompts)\n"
    cat_basins = {}
    for label, basin, tok in entries:
        cat_basins[basin] = cat_basins.get(basin, 0) + 1
    for b, c in sorted(cat_basins.items(), key=lambda x: -x[1]):
        md += f"- {b}: {c}/{len(entries)}\n"
    md += "\n"

if mismatches:
    md += "## Prediction Mismatches\n\n"
    for label, pred, actual in mismatches:
        md += f"- **{label}**: predicted `{pred}`, got `{actual}`\n"

# Save and display
with open(os.path.join(OUTPUT_DIR, 'hypothesis_assessment.md'), 'w') as f:
    f.write(md)
print(f"[SAVED] {OUTPUT_DIR}/hypothesis_assessment.md")
display(Markdown(md))

[SAVED] output/hypothesis_assessment.md


# Stage 1 Results: Hypothesis Assessment

| Prompt | Category | Predicted | Actual Terminal | Match? |
|:---|:---|:---|:---|:---|
| A01_physics | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A02_medical | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A03_neuro | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A04_climate | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A05_evolution | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A06_epistemology | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A07_sociology | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A08_linguistics | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A09_code | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A10_sql | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A11_ml | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A12_systems | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A13_networking | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A14_kant | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A15_sartre | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A16_wittgenstein | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A17_marx | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A18_gothic | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A19_romantic | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A20_modernist | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A21_dickens | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A22_legal | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A23_contract | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A24_patent | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| A25_academic_abs | Complex | `prolet` (high) | `D` -> **OTHER:D** | n |
| B01_napoleon | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B02_wwi | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B03_moon | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B04_rome | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B05_mlk | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B06_sources | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B07_breaking | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B08_editorial | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B09_sports | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B10_weather | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B11_alone | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B12_fear | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B13_joy | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B14_anger | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B15_casual | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B16_gossip | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B17_argument | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B18_advice | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B19_question | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| B20_reddit | Narrative | `prolet` (medium) | `D` -> **OTHER:D** | n |
| D01_water | Chemical | `unknown` (none) | `D` -> **OTHER:D** | n |
| D02_periodic | Chemical | `unknown` (none) | `D` -> **OTHER:D** | n |
| D03_organic | Chemical | `unknown` (none) | `D` -> **OTHER:D** | n |
| D04_equation | Chemical | `unknown` (none) | `D` -> **OTHER:D** | n |
| D05_amino | Chemical | `unknown` (none) | `D` -> **OTHER:D** | n |
| D06_physics_eq | Chemical | `unknown` (none) | `D` -> **OTHER:D** | n |
| D07_dna | Chemical | `unknown` (none) | `D` -> **OTHER:D** | n |
| D08_math | Chemical | `unknown` (none) | `D` -> **OTHER:D** | n |
| D09_units | Chemical | `unknown` (none) | `D` -> **OTHER:D** | n |
| D10_isotopes | Chemical | `unknown` (none) | `D` -> **OTHER:D** | n |
| E01_politics | Acronyms | `unknown` (none) | `D` -> **OTHER:D** | n |
| E02_tech | Acronyms | `unknown` (none) | `D` -> **OTHER:D** | n |
| E03_orgs | Acronyms | `unknown` (none) | `D` -> **OTHER:D** | n |
| E04_internet | Acronyms | `unknown` (none) | `D` -> **OTHER:D** | n |
| E05_finance | Acronyms | `unknown` (none) | `D` -> **OTHER:D** | n |
| E06_medical | Acronyms | `unknown` (none) | `D` -> **OTHER:D** | n |
| E07_military | Acronyms | `unknown` (none) | `D` -> **OTHER:D** | n |
| E08_academic | Acronyms | `unknown` (none) | `D` -> **OTHER:D** | n |
| E09_mixed | Acronyms | `unknown` (none) | `D` -> **OTHER:D** | n |
| E10_crypto | Acronyms | `unknown` (none) | `D` -> **OTHER:D** | n |
| C01_jack_jill | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C02_king_cole | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C03_mary_lamb | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C04_humpty | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C05_twinkle | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C06_dog | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C07_cat_mat | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C08_boy_girl | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C09_run | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C10_spot | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C11_genesis | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C12_beatitudes | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C13_psalm | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C14_commandment | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C15_fox_hen | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C16_ant_dove | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C17_tortoise | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C18_wolf | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C19_lion_mouse | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| C20_crow | Simple | `Divine` (low) | `D` -> **OTHER:D** | n |
| F01_anger | Vulgarity | `unknown` (none) | `D` -> **OTHER:D** | n |
| F02_insult | Vulgarity | `unknown` (none) | `D` -> **OTHER:D** | n |
| F03_frustration | Vulgarity | `unknown` (none) | `D` -> **OTHER:D** | n |
| F04_argument | Vulgarity | `unknown` (none) | `D` -> **OTHER:D** | n |
| F05_rant | Vulgarity | `unknown` (none) | `D` -> **OTHER:D** | n |
| F06_dismissal | Vulgarity | `unknown` (none) | `D` -> **OTHER:D** | n |
| F07_shock | Vulgarity | `unknown` (none) | `D` -> **OTHER:D** | n |
| F08_mild | Vulgarity | `unknown` (none) | `D` -> **OTHER:D** | n |
| F09_slur_adjacent | Vulgarity | `unknown` (none) | `D` -> **OTHER:D** | n |
| F10_exasperation | Vulgarity | `unknown` (none) | `D` -> **OTHER:D** | n |
| G01_punctuation | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G02_brackets | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G03_counting | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G04_fibonacci | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G05_primes | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G06_binary | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G07_the | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G08_period | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G09_space | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G10_newline | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G11_aaa | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G12_the_the | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G13_buffalo | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G14_nursery_acad | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G15_bible_code | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G16_nursery_vulgar | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G17_formal_slang | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G18_french | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G19_german | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G20_spanish | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G21_latin | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G22_japanese_rom | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G23_emoji | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G24_beatles | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G25_rickroll | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G26_bohemian | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G27_ignore | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G28_system | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G29_palindrome | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |
| G30_alphabet | Wild | `unknown` (none) | `D` -> **OTHER:D** | n |

---

## Basin Summary

| Basin | Count | % |
|:---|:---|:---|
| **OTHER:D** | 125 | 100.0% |

---

## Category Breakdown

### Complex (25 prompts)
- OTHER:D: 25/25

### Narrative (20 prompts)
- OTHER:D: 20/20

### Chemical (10 prompts)
- OTHER:D: 10/10

### Acronyms (10 prompts)
- OTHER:D: 10/10

### Simple (20 prompts)
- OTHER:D: 20/20

### Vulgarity (10 prompts)
- OTHER:D: 10/10

### Wild (30 prompts)
- OTHER:D: 30/30

## Prediction Mismatches

- **A01_physics**: predicted `prolet`, got `OTHER:D`
- **A02_medical**: predicted `prolet`, got `OTHER:D`
- **A03_neuro**: predicted `prolet`, got `OTHER:D`
- **A04_climate**: predicted `prolet`, got `OTHER:D`
- **A05_evolution**: predicted `prolet`, got `OTHER:D`
- **A06_epistemology**: predicted `prolet`, got `OTHER:D`
- **A07_sociology**: predicted `prolet`, got `OTHER:D`
- **A08_linguistics**: predicted `prolet`, got `OTHER:D`
- **A09_code**: predicted `prolet`, got `OTHER:D`
- **A10_sql**: predicted `prolet`, got `OTHER:D`
- **A11_ml**: predicted `prolet`, got `OTHER:D`
- **A12_systems**: predicted `prolet`, got `OTHER:D`
- **A13_networking**: predicted `prolet`, got `OTHER:D`
- **A14_kant**: predicted `prolet`, got `OTHER:D`
- **A15_sartre**: predicted `prolet`, got `OTHER:D`
- **A16_wittgenstein**: predicted `prolet`, got `OTHER:D`
- **A17_marx**: predicted `prolet`, got `OTHER:D`
- **A18_gothic**: predicted `prolet`, got `OTHER:D`
- **A19_romantic**: predicted `prolet`, got `OTHER:D`
- **A20_modernist**: predicted `prolet`, got `OTHER:D`
- **A21_dickens**: predicted `prolet`, got `OTHER:D`
- **A22_legal**: predicted `prolet`, got `OTHER:D`
- **A23_contract**: predicted `prolet`, got `OTHER:D`
- **A24_patent**: predicted `prolet`, got `OTHER:D`
- **A25_academic_abs**: predicted `prolet`, got `OTHER:D`
- **B01_napoleon**: predicted `prolet`, got `OTHER:D`
- **B02_wwi**: predicted `prolet`, got `OTHER:D`
- **B03_moon**: predicted `prolet`, got `OTHER:D`
- **B04_rome**: predicted `prolet`, got `OTHER:D`
- **B05_mlk**: predicted `prolet`, got `OTHER:D`
- **B06_sources**: predicted `prolet`, got `OTHER:D`
- **B07_breaking**: predicted `prolet`, got `OTHER:D`
- **B08_editorial**: predicted `prolet`, got `OTHER:D`
- **B09_sports**: predicted `prolet`, got `OTHER:D`
- **B10_weather**: predicted `prolet`, got `OTHER:D`
- **B11_alone**: predicted `prolet`, got `OTHER:D`
- **B12_fear**: predicted `prolet`, got `OTHER:D`
- **B13_joy**: predicted `prolet`, got `OTHER:D`
- **B14_anger**: predicted `prolet`, got `OTHER:D`
- **B15_casual**: predicted `prolet`, got `OTHER:D`
- **B16_gossip**: predicted `prolet`, got `OTHER:D`
- **B17_argument**: predicted `prolet`, got `OTHER:D`
- **B18_advice**: predicted `prolet`, got `OTHER:D`
- **B19_question**: predicted `prolet`, got `OTHER:D`
- **B20_reddit**: predicted `prolet`, got `OTHER:D`
- **C01_jack_jill**: predicted `Divine`, got `OTHER:D`
- **C02_king_cole**: predicted `Divine`, got `OTHER:D`
- **C03_mary_lamb**: predicted `Divine`, got `OTHER:D`
- **C04_humpty**: predicted `Divine`, got `OTHER:D`
- **C05_twinkle**: predicted `Divine`, got `OTHER:D`
- **C06_dog**: predicted `Divine`, got `OTHER:D`
- **C07_cat_mat**: predicted `Divine`, got `OTHER:D`
- **C08_boy_girl**: predicted `Divine`, got `OTHER:D`
- **C09_run**: predicted `Divine`, got `OTHER:D`
- **C10_spot**: predicted `Divine`, got `OTHER:D`
- **C11_genesis**: predicted `Divine`, got `OTHER:D`
- **C12_beatitudes**: predicted `Divine`, got `OTHER:D`
- **C13_psalm**: predicted `Divine`, got `OTHER:D`
- **C14_commandment**: predicted `Divine`, got `OTHER:D`
- **C15_fox_hen**: predicted `Divine`, got `OTHER:D`
- **C16_ant_dove**: predicted `Divine`, got `OTHER:D`
- **C17_tortoise**: predicted `Divine`, got `OTHER:D`
- **C18_wolf**: predicted `Divine`, got `OTHER:D`
- **C19_lion_mouse**: predicted `Divine`, got `OTHER:D`
- **C20_crow**: predicted `Divine`, got `OTHER:D`


### 5b. Cross-Prompt Convergence Matrix

In [7]:
# ============================================================
# VIS 5b: CROSS-PROMPT CONVERGENCE MATRIX
# ============================================================

labels = list(all_results.keys())
n = len(labels)
sim_matrix = np.zeros((n, n))

final_vectors = []
for label in labels:
    final_vec = all_results[label][-1]["mean_vector"]
    final_vectors.append(final_vec)

for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = torch.nn.functional.cosine_similarity(
            final_vectors[i].unsqueeze(0).float(),
            final_vectors[j].unsqueeze(0).float()
        ).item()

fig_sim = px.imshow(
    sim_matrix,
    x=labels, y=labels,
    color_continuous_scale="Viridis",
    title="Stage 1: Cross-Prompt Convergence (125 Prompts)",
    aspect="auto",
)
fig_sim.update_layout(template="plotly_dark", height=900, width=1200)
fig_sim.show()
fig_sim.write_image(os.path.join(OUTPUT_DIR, 'convergence_matrix.png'), scale=2)
print(f"[SAVED] {OUTPUT_DIR}/convergence_matrix.png")

off_diag = sim_matrix[np.triu_indices(n, k=1)]
print(f"\nMean cross-prompt similarity: {off_diag.mean():.4f}")
print(f"Min:  {off_diag.min():.4f}")
print(f"Max:  {off_diag.max():.4f}")

[SAVED] output/convergence_matrix.png

Mean cross-prompt similarity: 0.9982
Min:  0.9699
Max:  1.0000


### 5c. Dissolution Pathway Analysis

In [8]:
# ============================================================
# VIS 5c: DISSOLUTION PATHWAYS — Per Category
# ============================================================

md = "# Dissolution Pathways — Last-Token Top Prediction\n\n"

for cat_name, cat_dict in [
    ("Complex", COMPLEX), ("Narrative", NARRATIVE),
    ("Simple", SIMPLE), ("Chemical", CHEMICAL),
    ("Acronyms", ACRONYMS), ("Vulgarity", VULGARITY),
    ("Wild", WILD)
]:
    cat_labels = [k for k in cat_dict.keys() if k in all_results]
    if not cat_labels:
        continue
    
    md += f"## {cat_name} ({len(cat_labels)} prompts)\n\n"
    md += "| Iter | " + " | ".join(cat_labels[:10]) + " |\n"
    md += "| :--- | " + " | ".join([":---"] * min(len(cat_labels), 10)) + " |\n"
    
    for idx, iteration in enumerate(ITERATION_SCHEDULE):
        row = f"| **{iteration}** |"
        for label in cat_labels[:10]:
            snapshots = all_results[label]
            if idx < len(snapshots):
                tok = snapshots[idx]['top_tokens'][0][0]
                clean_t = tok.replace('\n', '↵').replace('`', "'").strip()
                row += f" `{clean_t}` |"
            else:
                row += " — |"
        md += row + "\n"
    md += "\n"

with open(os.path.join(OUTPUT_DIR, 'dissolution_pathways.md'), 'w', encoding='utf-8') as f:
    f.write(md)
print(f"[SAVED] {OUTPUT_DIR}/dissolution_pathways.md")
display(Markdown(md))

[SAVED] output/dissolution_pathways.md


# Dissolution Pathways — Last-Token Top Prediction

## Complex (25 prompts)

| Iter | A01_physics | A02_medical | A03_neuro | A04_climate | A05_evolution | A06_epistemology | A07_sociology | A08_linguistics | A09_code | A10_sql |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `the` | `that` | `learning` | `extinction` | `that` | `the` | `/` | `the` | `memory` | `'` |
| **2** | `the` | `that` | `the` | `the` | `that` | `the` | `/` | `the` | `D` | `D` |
| **3** | `local` | `that` | `local` | `local` | `local` | `the` | `national` | `local` | `D` | `D` |
| **5** | `local` | `local` | `D` | `local` | `D` | `local` | `D` | `local` | `D` | `D` |
| **10** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **20** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **50** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **100** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |

## Narrative (20 prompts)

| Iter | B01_napoleon | B02_wwi | B03_moon | B04_rome | B05_mlk | B06_sources | B07_breaking | B08_editorial | B09_sports | B10_weather |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `100` | `on` | `for` | `fifty` | `country` | `FBI` | `body` | `election` | `between` | `parts` |
| **2** | `the` | `"` | `.` | `tw` | `year` | `city` | `police` | `-` | `of` | `the` |
| **3** | `national` | `sh` | `local` | `national` | `game` | `MC` | `gun` | `national` | `of` | `local` |
| **5** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `national` | `local` | `local` |
| **10** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **20** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **50** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **100** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |

## Simple (20 prompts)

| Iter | C01_jack_jill | C02_king_cole | C03_mary_lamb | C04_humpty | C05_twinkle | C06_dog | C07_cat_mat | C08_boy_girl | C09_run | C10_spot |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `the` | `who` | `so` | `was` | `you` | `,` | `watch` | `hospital` | `can` | `pass` |
| **2** | `the` | `.` | `t` | `.` | `you` | `.` | `the` | `local` | `t` | `run` |
| **3** | `local` | `local` | `sh` | `D` | `we` | `D` | `the` | `GR` | `share` | `D` |
| **5** | `D` | `def` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **10** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **20** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **50** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **100** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |

## Chemical (10 prompts)

| Iter | D01_water | D02_periodic | D03_organic | D04_equation | D05_amino | D06_physics_eq | D07_dna | D08_math | D09_units | D10_isotopes |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `O` | `↵` | `↵` | `2` | `↵` | `=` | `G` | `the` | `s` | `Sr` |
| **2** | `D` | `m` | `MA` | `D` | `q` | `D` | `G` | `the` | `D` | `U` |
| **3** | `RAW` | `Q` | `AB` | `D` | `D` | `D` | `AB` | `local` | `D` | `D` |
| **5** | `D` | `D` | `D` | `D` | `D` | `D` | `AB` | `D` | `D` | `D` |
| **10** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **20** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **50** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **100** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |

## Acronyms (10 prompts)

| Iter | E01_politics | E02_tech | E03_orgs | E04_internet | E05_finance | E06_medical | E07_military | E08_academic | E09_mixed | E10_crypto |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `BR` | `↵` | `FBI` | `I` | `↵` | `OR` | `↵` | `ACT` | `S` | `↵` |
| **2** | `BR` | `RAW` | `D` | `BR` | `D` | `D` | `MA` | `D` | `D` | `D` |
| **3** | `BR` | `RAW` | `D` | `BR` | `D` | `D` | `MA` | `D` | `D` | `D` |
| **5** | `D` | `D` | `D` | `AB` | `D` | `D` | `AB` | `D` | `D` | `D` |
| **10** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **20** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **50** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **100** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |

## Vulgarity (10 prompts)

| Iter | F01_anger | F02_insult | F03_frustration | F04_argument | F05_rant | F06_dismissal | F07_shock | F08_mild | F09_slur_adjacent | F10_exasperation |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `fucking` | `to` | `have` | `fucking` | `should` | `about` | `?` | `thing` | `for` | `you` |
| **2** | `we` | `"` | `we` | `.` | `is` | `of` | `.` | `est` | `of` | `you` |
| **3** | `sh` | `"` | `share` | `AB` | `I` | `of` | `D` | `we` | `of` | `local` |
| **5** | `D` | `D` | `def` | `D` | `D` | `D` | `D` | `D` | `def` | `D` |
| **10** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **20** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **50** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **100** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |

## Wild (30 prompts)

| Iter | G01_punctuation | G02_brackets | G03_counting | G04_fibonacci | G05_primes | G06_binary | G07_the | G08_period | G09_space | G10_newline |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `;` | `|` | `10` | `100` | `31` | `01` | `first` | `↵` | `` | `I` |
| **2** | `#` | `A` | `.` | `D` | `D` | `01` | `story` | `0` | `I` | `I` |
| **3** | `D` | `D` | `.` | `D` | `D` | `AB` | `MC` | `D` | `D` | `A` |
| **5** | `D` | `D` | `D` | `D` | `D` | `AB` | `D` | `D` | `D` | `D` |
| **10** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **20** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **50** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |
| **100** | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` | `D` |



### 5d. Sentence Dissolution Tables — Full Position Reconstruction

In [9]:
# ============================================================
# VIS 5d: SENTENCE DISSOLUTION TABLES
# ============================================================

md = "# Full Sentence Dissolution — All 125 Prompts\n\n"

for label in PROMPT_LIBRARY.keys():
    if label not in all_results:
        continue
    snapshots = all_results[label]
    predicted, conf = PREDICTIONS[label]
    category = CATEGORY_MAP[label]
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    
    md += f"### {label} [{category}] -> `{terminal}` (predicted: `{predicted}`)\n"
    md += f"*\"{PROMPT_LIBRARY[label]}\"*\n\n"
    md += "| Iter | Reconstructed Output |\n"
    md += "|:---|:---|\n"
    for s in snapshots:
        tokens = s['all_position_tokens']
        clean = [t.replace('\n', '↵').replace('|', '\\|') for t in tokens]
        sentence = ' '.join(clean)
        md += f"| {s['iteration']} | {sentence} |\n"
    md += "\n"

with open(os.path.join(OUTPUT_DIR, 'dissolution_sentences.md'), 'w', encoding='utf-8') as f:
    f.write(md)
print(f"[SAVED] {OUTPUT_DIR}/dissolution_sentences.md")
print(f"Full dissolution tables: {len(all_results)} prompts written.")

[SAVED] output/dissolution_sentences.md
Full dissolution tables: 125 prompts written.


### 5e. 3D PCA Trajectories — All Prompts

In [10]:
# ============================================================
# VIS 5e: 3D PCA TRAJECTORIES
# ============================================================
from sklearn.decomposition import PCA
import pandas as pd

all_vecs = []
labels_list = []
cats_list = []
iters_list = []
text_list = []

for label, snapshots in all_results.items():
    for s in snapshots:
        all_vecs.append(s["mean_vector"].detach().cpu().numpy())
        labels_list.append(label)
        cats_list.append(CATEGORY_MAP.get(label, 'Unknown'))
        iters_list.append(s["iteration"])
        top_tok = s['top_tokens'][0][0].replace('\n', '↵').strip()
        text_list.append(f"Iter {s['iteration']}: {top_tok}")

all_vecs = np.array(all_vecs)
pca = PCA(n_components=3)
vecs_3d = pca.fit_transform(all_vecs)

df = pd.DataFrame({
    'x': vecs_3d[:, 0],
    'y': vecs_3d[:, 1],
    'z': vecs_3d[:, 2],
    'Prompt': labels_list,
    'Category': cats_list,
    'Iteration': iters_list,
    'Top_Token': text_list
})

# Color by category for readability
fig_topo = px.line_3d(
    df, x='x', y='y', z='z',
    color='Category',
    hover_name='Top_Token',
    markers=True,
    title=f"Stage 1: Attractor Landscape — {len(all_results)} Prompt Trajectories<br>"
          f"<sup>(Explained Variance: {sum(pca.explained_variance_ratio_)*100:.1f}%)</sup>"
)
fig_topo.update_traces(marker=dict(size=3), line=dict(width=2))
fig_topo.update_layout(
    template="plotly_dark",
    height=900,
    width=1200,
    scene=dict(
        xaxis_title="PC 1",
        yaxis_title="PC 2",
        zaxis_title="PC 3",
    )
)
fig_topo.show()
fig_topo.write_image(os.path.join(OUTPUT_DIR, 'topology_3d.png'), scale=2)
print(f"[SAVED] {OUTPUT_DIR}/topology_3d.png")

[SAVED] output/topology_3d.png


### 5f. Basin Distribution Chart

In [11]:
# ============================================================
# VIS 5f: BASIN DISTRIBUTION BAR CHART
# ============================================================

basin_data = []
for label, snapshots in all_results.items():
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    category = CATEGORY_MAP.get(label, 'Unknown')
    if terminal == 'prolet':
        basin = 'prolet'
    elif terminal == 'Divine':
        basin = 'Divine'
    else:
        basin = terminal
    basin_data.append({'Prompt': label, 'Category': category, 'Basin': basin})

basin_df = pd.DataFrame(basin_data)
basin_summary = basin_df.groupby(['Category', 'Basin']).size().reset_index(name='Count')

fig_basin = px.bar(
    basin_summary, x='Category', y='Count', color='Basin',
    title=f"Stage 1: Basin Distribution by Category ({len(all_results)} prompts)",
    barmode='stack'
)
fig_basin.update_layout(template="plotly_dark", height=500, width=900)
fig_basin.show()
fig_basin.write_image(os.path.join(OUTPUT_DIR, 'basin_distribution.png'), scale=2)
print(f"[SAVED] {OUTPUT_DIR}/basin_distribution.png")

[SAVED] output/basin_distribution.png


In [12]:
# ============================================================
# STEP 6: SAVE RAW DATA
# ============================================================

save_data = {}
for label, snapshots in all_results.items():
    save_data[label] = {
        "iterations": [s["iteration"] for s in snapshots],
        "last_vectors": torch.stack([s["last_vector"] for s in snapshots]),
        "mean_vectors": torch.stack([s["mean_vector"] for s in snapshots]),
        "last_norms": [s["last_norm"] for s in snapshots],
        "mean_norms": [s["mean_norm"] for s in snapshots],
        "cosine_sims_last": [s["cosine_sim_last"] for s in snapshots],
        "cosine_sims_mean": [s["cosine_sim_mean"] for s in snapshots],
        "position_similarity": [s["position_similarity"] for s in snapshots],
        "top_tokens": [s["top_tokens"] for s in snapshots],
        "all_position_tokens": [s["all_position_tokens"] for s in snapshots],
    }

torch.save(save_data, os.path.join(OUTPUT_DIR, 'stage1_results.pt'))
print(f"[SAVED] {OUTPUT_DIR}/stage1_results.pt")

config = {
    "schedule": ITERATION_SCHEDULE,
    "layer_start": LAYER_START,
    "layer_end": LAYER_END,
    "prompt_count": len(PROMPT_LIBRARY),
    "model": "gpt2-medium",
    "mode": "stage1_attractor_dominance",
}
torch.save(config, os.path.join(OUTPUT_DIR, 'stage1_config.pt'))
print(f"[SAVED] {OUTPUT_DIR}/stage1_config.pt")
print(f"\ny All artifacts saved to {os.path.abspath(OUTPUT_DIR)}")

[SAVED] output/stage1_results.pt
[SAVED] output/stage1_config.pt

y All artifacts saved to c:\Users\Fab2\Desktop\AI\_learn\_fold\03_MECHINIPHYLUM\_LAB_NOTEBOOKS\lucier-repo\experiments\gpt2_medium\output
